<a href="https://colab.research.google.com/github/jyryu3161/2026CADD/blob/main/week11_ADMET_AI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 도킹 결과에 ADMET 예측값 붙이기

**실습 목표**　각 후보물질의 SMILES로 흡수·분포·대사·배설·독성(ADMET)을 예측하고, 도킹 점수와 같은 행에서 비교합니다.

Colab의 기본 CPU 런타임에서 **런타임 → 모두 실행**을 누르고 파일 선택 창에서 `scores.csv`를 올리세요. 마지막 셀이 원본 15개 열 오른쪽에 결과를 붙인 `scores_admet.csv`를 다운로드합니다.

**사용 모델**　ADMET-AI v2.0.1. [공식 저장소 및 모델 설명](https://github.com/swansonk14/admet_ai/tree/v_2.0.1)

## 1. ADMET-AI 설치

Colab 안에 실습용 Python 3.12 환경을 만들고, 공식 GitHub 저장소의 v2.0.1 코드와 학습된 모델을 설치합니다. 첫 실행에는 패키지 다운로드 시간이 필요합니다.

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = [sys.executable, "-m", "uv"]
env_dir = Path("/content/admet-ai-venv")
admet_python = env_dir / "bin" / "python"
subprocess.run(uv + ["venv", "--allow-existing", "--python", "3.12", str(env_dir)], check=True)

revision = "c65bf0418e19c65d7228f9e40da5d0152aade756"  # 공식 태그 v_2.0.1
source_url = f"https://github.com/swansonk14/admet_ai/archive/{revision}.zip"
subprocess.run(uv + [
    "pip", "install", "--python", str(admet_python),
    "--find-links", "https://download.pytorch.org/whl/cpu/torch/",
    "torch==2.10.0+cpu", "chemprop==2.2.2", "lightning==2.6.1",
    f"admet-ai @ {source_url}",
], check=True)
print("ADMET-AI 설치 완료")

## 2. 도킹 결과 CSV 업로드

첨부된 `scores.csv`에는 2,000행이 있습니다. 미리보기에서 첫 행의 `CID`, 도킹 점수, SMILES를 확인하세요.

In [ ]:
from tempfile import mkdtemp

import pandas as pd
from google.colab import files
from IPython.display import display

work_dir = Path(mkdtemp(prefix="admet-", dir="/content"))
uploaded = files.upload(target_dir=str(work_dir))
if len(uploaded) != 1:
    raise ValueError("CSV 파일 한 개를 선택하세요.")
input_path = work_dir / Path(next(iter(uploaded))).name
output_path = work_dir / f"{input_path.stem}_admet.csv"
read_options = dict(dtype=str, na_filter=False, encoding="utf-8-sig")
original = pd.read_csv(input_path, **read_options)
if not {"normalized_SMILES", "SMILES"}.intersection(original.columns):
    raise ValueError("normalized_SMILES 또는 SMILES 열을 확인하세요.")

print(f"{input_path.name} · {len(original):,}행 × {len(original.columns)}열")
preview = ["row_number", "CID", "affinity_kcal_mol", "normalized_SMILES", "SMILES"]
display(original[[c for c in preview if c in original]].head(3))

## 3. 분자별 예측 및 원본 행에 연결

각 행의 `normalized_SMILES`를 RDKit으로 확인하고, 비었거나 해석할 수 없으면 같은 행의 `SMILES`를 확인합니다. 동일한 SMILES는 한 번만 예측한 뒤 해당하는 모든 행에 결과를 붙입니다. 구조가 누락되거나 잘못된 행은 예측값을 비워 두고 `ADMET_status`에 이유를 기록합니다.

첫 셀은 실습용 환경에서 실행할 코드를 준비하고, 다음 셀은 업로드한 파일에 적용합니다.

In [ ]:
%%writefile /content/predict_admet.py
import sys
from pathlib import Path

import pandas as pd
from rdkit import Chem, rdBase
from admet_ai import ADMETModel
from admet_ai.constants import DEFAULT_ADMET_PATH

input_path, output_path = map(Path, sys.argv[1:])
# CID의 앞자리 0, 도킹 점수의 소수 자릿수, 빈 칸을 그대로 보존합니다.
data = pd.read_csv(input_path, dtype=str, na_filter=False, encoding="utf-8-sig")
columns = [name for name in ("normalized_SMILES", "SMILES") if name in data]
if not columns:
    raise ValueError("입력 CSV에 normalized_SMILES 또는 SMILES 열이 필요합니다.")


def select_smiles(values):
    """한 행에서 RDKit이 해석할 수 있는 첫 번째 구조와 그 출처를 찾습니다."""
    has_text = False
    for column, raw in zip(columns, values):
        smiles = raw.strip()
        if smiles:
            has_text = True
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None and mol.GetNumAtoms() > 0:
                return smiles, column, "ok"
    return "", "", "invalid_smiles" if has_text else "missing_smiles"


with rdBase.BlockLogs():
    selected = [select_smiles(row) for row in data[columns].itertuples(index=False, name=None)]
inputs = pd.DataFrame(selected, columns=["smiles", "ADMET_input_column", "ADMET_status"])
unique_smiles = inputs.loc[inputs["ADMET_status"].eq("ok"), "smiles"].unique().tolist()

if unique_smiles:
    print(f"고유 SMILES {len(unique_smiles):,}개 예측 중", flush=True)
    model = ADMETModel(num_workers=0)
    predictions = model.predict(smiles=unique_smiles)
    if not predictions.index.is_unique or set(predictions.index) != set(unique_smiles):
        raise RuntimeError("요청한 SMILES와 모델이 반환한 SMILES가 일치하지 않습니다.")
else:
    # 모두 빈 값이거나 잘못된 구조여도 같은 형식의 결과 파일을 만듭니다.
    properties = pd.read_csv(DEFAULT_ADMET_PATH)["id"].tolist()
    percentiles = [name + "_drugbank_approved_percentile" for name in properties]
    predictions = pd.DataFrame(columns=properties + percentiles)

# 모델의 SMILES 인덱스로 각 입력 행을 찾아 중복 구조와 예측 불가 행도 보존합니다.
aligned = predictions.reindex(inputs["smiles"]).reset_index(drop=True).add_prefix("ADMET_")
added = pd.concat([inputs[["ADMET_status", "ADMET_input_column"]], aligned], axis=1)
overlap = data.columns.intersection(added.columns)
if len(overlap):
    raise ValueError(f"입력에 결과 열이 이미 있습니다. 최초 입력 CSV를 선택하세요. {list(overlap)}")

result = pd.concat([data, added], axis=1)
assert result.iloc[:, :data.shape[1]].equals(data)
result.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"완료 · 입력 {len(data):,}행 · 예측 성공 {inputs['ADMET_status'].eq('ok').sum():,}행")

In [ ]:
import os

print(f"{len(original):,}행의 예측을 시작합니다.", flush=True)
log_path = work_dir / "prediction.log"
with log_path.open("w", encoding="utf-8") as log:
    run = subprocess.run(
        [str(admet_python), "/content/predict_admet.py", str(input_path), str(output_path)],
        stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, "OMP_NUM_THREADS": "2", "MPLBACKEND": "Agg"},
    )
if run.returncode:
    print(log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
    run.check_returncode()

result = pd.read_csv(output_path, **read_options)
assert result.iloc[:, :original.shape[1]].equals(original), "원본 행 또는 열이 달라졌습니다."
print(f"결과 {len(result):,}행 × {len(result.columns)}열")
display(result["ADMET_status"].value_counts().rename("행 수").to_frame())
preview = ["CID", "affinity_kcal_mol", "ADMET_status", "ADMET_HIA_Hou",
           "ADMET_Solubility_AqSolDB", "ADMET_hERG"]
display(result[[c for c in preview if c in result]].head())

## 4. 결과 CSV 다운로드

이 셀을 실행하면 다운로드가 시작됩니다. 다운로드를 다시 시작하려면 이 셀만 한 번 더 실행하세요.

In [ ]:
files.download(str(output_path))

### 결과에서 읽어볼 열

| 열 | 읽는 방법 |
|---|---|
| `ADMET_status` · `ADMET_input_column` | 예측 성공은 `ok`로 표시되고 실제 사용한 구조 열이 함께 기록됩니다. `missing_smiles`는 구조 누락, `invalid_smiles`는 구조 해석 실패입니다. |
| `ADMET_HIA_Hou` | 사람 장 흡수 분류 모델의 양성 예측 확률입니다. 값이 1에 가까울수록 흡수 양성으로 예측합니다. |
| `ADMET_Solubility_AqSolDB` | 예측 수용해도입니다. 단위는 log₁₀(mol/L)이며 값이 클수록 더 잘 녹는다는 뜻입니다. |
| `ADMET_hERG` | hERG 칼륨 채널 차단에 대한 양성 예측 확률입니다. |
| `…_drugbank_approved_percentile` | 같은 지표를 DrugBank 승인 약물과 비교한 백분위입니다. 예를 들어 90은 해당 값이 참조 분포의 약 90번째 백분위에 위치한다는 뜻입니다. |

**실습 질문**　`affinity_kcal_mol`이 가장 낮은 후보 5개의 예측 수용해도와 hERG 차단 확률을 나란히 비교해 보세요. [공식 지표별 정의·단위](https://github.com/swansonk14/admet_ai/blob/v_2.0.1/admet_ai/resources/data/admet.csv)